# 02 — SQL analysis with SQLite

The database is built from the cleaned CSV headers. The queries use the confirmed TMDB columns in this project and can be run one at a time. See `sql/analysis_queries.sql` for 27 documented questions.

In [ ]:
from pathlib import Path
import sys
import sqlite3
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.clean_data import clean_all
from src.database import create_database, DATABASE_PATH
if not (ROOT / 'data' / 'processed' / 'movies.csv').exists():
    clean_all(ROOT / 'data' / 'raw', ROOT / 'data' / 'processed', require_all=True)
create_database(ROOT / 'data' / 'processed', DATABASE_PATH)

## How many movies are there?
This count comes directly from SQLite, not a hard-coded number. The result reflects the cleaned movie table.

In [ ]:
with sqlite3.connect(DATABASE_PATH) as connection:
    movie_count = pd.read_sql_query('SELECT COUNT(*) AS movie_count FROM movies', connection)
display(movie_count)

## Which genres occur most frequently?
A movie can belong to multiple genres. This query counts distinct movies within each genre, not genre links across all rows.

In [ ]:
genre_query = '''
SELECT g.genre_name, COUNT(DISTINCT mg.movie_id) AS movie_count
FROM genres g JOIN movie_genres mg ON g.genre_id = mg.genre_id
GROUP BY g.genre_id, g.genre_name ORDER BY movie_count DESC
'''
with sqlite3.connect(DATABASE_PATH) as connection:
    genre_counts = pd.read_sql_query(genre_query, connection)
display(genre_counts.head(15))

## Which directors have the most titles?
Crew records include several jobs. The query filters to rows whose job is `Director`; it does not infer a director from another crew role.

In [ ]:
director_query = '''
SELECT person_name AS director, COUNT(DISTINCT movie_id) AS movie_count
FROM crew WHERE lower(job) = 'director'
GROUP BY person_id, person_name ORDER BY movie_count DESC LIMIT 15
'''
with sqlite3.connect(DATABASE_PATH) as connection:
    directors = pd.read_sql_query(director_query, connection)
display(directors)